## 1、何时写入记忆

官方介绍了两种方式。

### 1.1 在主流程里写（hot path）

也就是：用户发消息，ai一边回答，一边决定要不要记下来。

优点：
- 立即生效。
- 下一轮马上能用。
- 用户可感知，透明。

缺点：
- 增加延迟。
- 逻辑变复杂。

### 1.2 在后台写（background）

就是先回答用户，记忆整理放后台异步做。

优点：
- 主流程更快。
- 记忆逻辑更独立。
- 更适合批量整理。

缺点：
- 不能立刻生效
- 要决定多久整理一次
- 触发时机不好选

工程上通常这么选：
- 用户偏好、账号资料 ：可热路径写
- 对话摘要、经验沉淀、行为分析 ：更适合后台写

## 2、Static runtime Context学习

静态运行时上下文(Static runtime Context)表示不可变的数据，如用户元数据、工具和传递给应用程序的数据库连接对象。通常在运行开始时通过 invoke / stream 的 context 参数传递，此类数据在运行期间不会更改。

### 2.1 在中间件访问

#### 2.1.1 Node-style hooks

此类钩子都通过 runtime.context 访问上下文对象。用户自定义 ContextSchema 用 @dataclass 修饰，在Agent创建时通过 context_schema 参数传递。

本例从长期记忆中查询用户额度，如果额度用尽则中断流程。

In [1]:
import os
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langgraph.prebuilt.tool_node import ToolCallRequest
from langgraph.types import Command
from loguru import logger
from dataclasses import dataclass
from typing import Any

from dotenv import load_dotenv
from langchain.agents.middleware import AgentMiddleware, hook_config, ExtendedModelResponse
from langchain.agents.middleware.types import StateT, ResponseT
from langchain_core.messages import AIMessage, ToolMessage
from langgraph.runtime import Runtime
from langgraph.store.memory import InMemoryStore
from langgraph.typing import ContextT

load_dotenv(override=True)

store = InMemoryStore()

namespace = ("users", "credits")
key1 = 'Ada Lovelace'
value1 = {
    "tokens_credit_left": 5000,
    "user_level": 5
}

key2 = 'Blackwell'
value2 = {
    "tokens_credit_left": 2999,
    "user_level": 5
}

key3 = 'Ampere'
value3 = {
    "tokens_credit_left": 1000,
    "user_level": 5
}

store.put(namespace, key1, value1)
store.put(namespace, key2, value2)
store.put(namespace, key3, value3)

@dataclass
class UserContext:
    username: str

class CheckCredit(AgentMiddleware):
    @hook_config(can_jump_to=["end"])
    def before_model(self, state: StateT, runtime: Runtime[ContextT]) -> dict[str, Any] | None:
        store = runtime.store
        context = runtime.context

        username = context.username
        credit = store.get(namespace, username)
        if not credit:
            return {
                "jump_to": "end",
                "messages": AIMessage("您尚未注册")
            }
        if credit.value["tokens_credit_left"] < 3000:
            return {
                "jump_to": "end",
                "messages": AIMessage(f"{username}额度不足，请充值")
            }

        return None

    def after_model(self, state: StateT, runtime: Runtime[ContextT]) -> dict[str, Any] | None:
        store = runtime.store
        context = runtime.context
        username = context.username
        credit = store.get(namespace, username)
        usage_metadata = state["messages"][-1].usage_metadata
        token_usage = usage_metadata["input_tokens"] + usage_metadata["output_tokens"] * 6
        credit.value["tokens_credit_left"] -= token_usage

        store.put(namespace, username, credit.value)

        return None

    def after_agent(self, state: StateT, runtime: Runtime[ContextT]) -> dict[str, Any] | None:
        store = runtime.store
        context = runtime.context
        username = context.username
        item = store.get(namespace, username)
        credit = item.value["tokens_credit_left"]
        logger.info(f"{username} 当前剩余额度：{credit}")
        return None

model = init_chat_model(
    model="deepseek-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
)

agent = create_agent(
    model=model,
    middleware=[CheckCredit()],
    store=store,
    context_schema=UserContext
)

response1 = agent.invoke(
    {"messages": ["你好啊，你知道 Ada Lovelace 的贡献吗？"]},
    context=UserContext(username="Ada Lovelace")
)
print("=" * 30, '-> Ada Lovelace messages <-', '=' * 30)
for msg in response1["messages"]:
    msg.pretty_print()

response2 = agent.invoke(
    {"messages": ["你好啊，你知道 Blackwell 的贡献吗？"]},
    context=UserContext(username="Blackwell")
)
print("=" * 30, '-> Blackwell messages <-', '=' * 30)
for msg in response2["messages"]:
    msg.pretty_print()

2026-11-23 22:32:29.539 +08:00 | INFO     | __main__:after_agent:87 - Ada Lovelace 当前剩余额度：2167
2026-11-23 22:32:29.542 +08:00 | INFO     | __main__:after_agent:87 - Blackwell 当前剩余额度：2999


============================== -> Ada Lovelace messages <- ==============================
================================ Human Message =================================

你好啊，你知道 Ada Lovelace 的贡献吗？
================================== Ai Message ==================================

你好！我知道 Ada Lovelace（奥古斯塔·阿达·洛夫莱斯，1815–1852）是19世纪英国数学家，也是计算史上非常重要的人物。她的主要贡献包括：

1. **与查尔斯·巴贝奇合作研究“分析机”**  
   巴贝奇设计了差分机和分析机，而 Ada Lovelace 深入理解了分析机的潜力，并为其写了详细注释。

2. **翻译并大幅增补 Menabrea 的论文**  
   她翻译了意大利工程师 Luigi Menabrea 关于分析机的文章，并加入了自己的长篇“注释”。这些注释比原文影响更大，其中包含了对分析机原理和用途的深入解释。

3. **提出著名的“Note G”算法**  
   在注释 G 中，她给出了用分析机计算伯努利数（Bernoulli numbers）的详细步骤。这常被视为历史上最早发表的计算机算法/程序之一，因此她常被称为“第一位程序员”。

4. **预见到计算机不只用于数值计算**  
   她指出分析机不仅能处理数字，还可能处理符号、文字、音乐等，只要这些对象能被抽象表示和操作。这体现了很强的“通用计算”思想。

5. **区分“计算”与“创造”**  
   她也强调机器只能按照人给的规则执行，不能自行产生原创思想。这个观点在今天讨论人工智能时仍常被引用。

总的来说，Ada Lovelace 的贡献不只是“写了第一个程序”，更在于她早期洞察了可编程机器的潜力、算法表达方式以及计算的通用性。她因此成为计算机科学和女性科技史上的标志性人物。
============================== -> Blackwell messages <- ===========

#### 2.1.2 Wrap-style hooks

1、wrap_model_call

本例通过 context 中记录的用户身份信息调整暴露给模型的工具集。

此处更改模型请求的方法是 transient request update ，见上文短期记忆 2-4-2

In [4]:
from dataclasses import dataclass
from typing import Callable
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, ModelResponse,ModelRequest
from langchain.messages import AIMessage
from langchain.tools import tool
from langgraph.store.memory import InMemoryStore
from dotenv import load_dotenv
from loguru import logger
from langchain.agents.middleware.types import ResponseT, ExtendedModelResponse

load_dotenv()
store = InMemoryStore()

namespace = ("users", "credits")
key1 = 'Ada Lovelace'
value1 = {
    "tokens_left": 3000, # 剩余大模型token额度
    "get_weather": "yes", # 是否认证天气查询服务
    "get_news": "no" # 是否认证网络爬虫服务，用于查询新闻
}

key2 = 'Blackwell'
value2 = {
    "tokens_left": 2000, # 剩余大模型token额度
    "get_weather": "no", # 是否认证天气查询服务
    "get_news": "yes" # 是否认证网络爬虫服务，用于查询新闻
}

key3 = 'Ampere'
value3 = {
    "tokens_left": 3000, # 剩余大模型token额度
    "get_weather": "yes", # 是否认证天气查询服务
    "get_news": "yes" # 是否认证网络爬虫服务，用于查询新闻
}

store.put(namespace, key1, value1)
store.put(namespace, key2, value2)
store.put(namespace, key3, value3)

@tool
def get_weather(city: str):
    """
    查询指定城市当日天气

    Args:
        city: 城市名称
    """
    return f"{city} 今天天气不错"

@tool()
def get_news():
    """查询当日新闻"""
    return "美伊尚未达成停战协议"

@dataclass
class UserContext:
    username: str

class CheckCredit(AgentMiddleware):
    def wrap_model_call(
        self,
        request: ModelRequest[ContextT],
        handler: Callable[[ModelRequest[ContextT]], ModelResponse[ResponseT]],
    ) -> ModelResponse[ResponseT] | AIMessage | ExtendedModelResponse[ResponseT]:
        context = request.runtime.context
        store = request.runtime.store

        value = store.get(namespace, context.username).value
        tools = []

        for tool in request.tools:
            tool_name = tool.name
            if value[tool_name] == "yes":
                tools.append(tool)
            else:
                logger.warning(f"{context.username} 无权调用 {tool_name}")

        # 更改调用请求携带的工具集，仅本次生效
        request = request.override(tools=tools)
        return handler(request)

agent = create_agent(
    model = model,
    middleware=[CheckCredit()],
    store=store,
    tools=[get_weather, get_news],
    context_schema=UserContext
)

response1 = agent.invoke(
    {"messages": ["你好啊，今天北京天气如何，今日有哪些新闻？"]},
    context=UserContext(username="Ada Lovelace")
)
print("=" * 30, '-> Ada Lovelace messages <-', '=' * 30)
for msg in response1["messages"]:
    msg.pretty_print()

response2 = agent.invoke(
    {"messages": ["你好啊，今天北京天气如何，今日有哪些新闻？"]},
    context=UserContext(username="Blackwell")
)
print("=" * 30, '-> Blackwell messages <-', '=' * 30)
for msg in response2["messages"]:
    msg.pretty_print()

response3 = agent.invoke(
    {"messages": ["你好啊，今天北京天气如何，今日有哪些新闻？"]},
    context=UserContext(username="Ampere")
)
print("=" * 30, '-> Ampere messages <-', '=' * 30)
for msg in response3["messages"]:
    msg.pretty_print()

2026-11-23 22:43:27.968 +08:00 | WARNING  | __main__:wrap_model_call:77 - Ada Lovelace 无权调用 get_news
2026-11-23 22:43:28.878 +08:00 | WARNING  | __main__:wrap_model_call:77 - Ada Lovelace 无权调用 get_news
2026-11-23 22:43:29.800 +08:00 | WARNING  | __main__:wrap_model_call:77 - Blackwell 无权调用 get_weather


============================== -> Ada Lovelace messages <- ==============================
================================ Human Message =================================

你好啊，今天北京天气如何，今日有哪些新闻？
================================== Ai Message ==================================

我来帮你查一下北京的天气～
Tool Calls:
  get_weather (call_00_hUuhIpLdPxpwkaVGoluc6216)
 Call ID: call_00_hUuhIpLdPxpwkaVGoluc6216
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

北京 今天天气不错
================================== Ai Message ==================================

你好呀！👋

**北京今日天气**：今天天气不错 ☀️，适合出门活动～

不过关于**今日新闻**，我这边暂时没有获取新闻的工具，所以没办法帮你查实时新闻资讯。建议你可以看看：
- 新闻类 App（如新华社、人民日报、澎湃新闻等）
- 或直接在搜索引擎搜「今日新闻」

如果你需要查其他城市的天气，随时告诉我～ 😊


2026-11-23 22:43:30.722 +08:00 | WARNING  | __main__:wrap_model_call:77 - Blackwell 无权调用 get_weather


============================== -> Blackwell messages <- ==============================
================================ Human Message =================================

你好啊，今天北京天气如何，今日有哪些新闻？
================================== Ai Message ==================================

你好！我先帮你查一下今天的新闻。

不过要说明一下：我目前**没有查询天气的工具**，所以无法给你北京今天的实时天气。建议你打开手机天气 App 或天气网站查看。

现在我去查新闻 👇
Tool Calls:
  get_news (call_00_UIJ8hDCAzndDd1E6ExOG8308)
 Call ID: call_00_UIJ8hDCAzndDd1E6ExOG8308
  Args:
================================= Tool Message =================================
Name: get_news

美伊尚未达成停战协议
================================== Ai Message ==================================

我查到了，不过返回的新闻只有一条，内容比较少：

**今日新闻**
- 美伊尚未达成停战协议

这条看起来是国际新闻，但工具只返回了标题级别的信息，没有更多细节（比如是哪一轮谈判、时间地点、双方表态等）。如果你想了解详情，我可以帮你分析这个话题的背景，或者你可以查阅权威媒体的报道。

关于**北京天气**，再提醒一下：我这边没有天气查询工具，没办法给你实时数据。你可以看看手机自带的天气应用，那个最准。

需要我针对美伊局势展开讲讲背景吗？
============================== -> Ampere messages <- ==============================
=============================

2、wrap_tool_call

在上述案例的基础上，我们补充额度校验，和长期记忆更新。

In [8]:
from dataclasses import dataclass
from typing import Callable
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, ModelResponse,ModelRequest
from langchain.messages import AIMessage
from langchain.tools import tool
from langgraph.store.memory import InMemoryStore
from dotenv import load_dotenv
from loguru import logger
from langchain.agents.middleware.types import ResponseT, ExtendedModelResponse
from langgraph.prebuilt.tool_node import ToolCallRequest
from langchain_core.messages.tool import ToolMessage
from langgraph.types import Command

load_dotenv()
store = InMemoryStore()

namespace = ("users", "credits")
key1 = 'Ada Lovelace'
value1 = {
    "tokens_left": 3000, # 剩余大模型token额度
    "get_weather": "yes", # 是否认证天气查询服务
    "get_news": "no" # 是否认证网络爬虫服务，用于查询新闻
}

key2 = 'Blackwell'
value2 = {
    "tokens_left": 2000, # 剩余大模型token额度
    "get_weather": "no", # 是否认证天气查询服务
    "get_news": "yes" # 是否认证网络爬虫服务，用于查询新闻
}

key3 = 'Ampere'
value3 = {
    "tokens_left": 3000, # 剩余大模型token额度
    "get_weather": "yes", # 是否认证天气查询服务
    "get_news": "yes" # 是否认证网络爬虫服务，用于查询新闻
}

store.put(namespace, key1, value1)
store.put(namespace, key2, value2)
store.put(namespace, key3, value3)

# 工具额度消耗
CREDIT_MAP = {
    "get_weather": 10,
    "get_news": 20
}

@tool
def get_weather(city: str):
    """
    查询指定城市当日天气

    Args:
        city: 城市名称
    """
    return f"{city} 今天天气不错"

@tool()
def get_news():
    """查询当日新闻"""
    return "美伊尚未达成停战协议"

@dataclass
class UserContext:
    username: str

class CheckCredit(AgentMiddleware):
    def wrap_model_call(
        self,
        request: ModelRequest[ContextT],
        handler: Callable[[ModelRequest[ContextT]], ModelResponse[ResponseT]],
    ) -> ModelResponse[ResponseT] | AIMessage | ExtendedModelResponse[ResponseT]:
        context = request.runtime.context
        store = request.runtime.store

        value = store.get(namespace, context.username).value
        tools = []

        for tool in request.tools:
            tool_name = tool.name
            if value[tool_name] == "yes":
                tools.append(tool)
            else:
                logger.warning(f"{context.username} 无权调用 {tool_name}")

        # 更改调用请求携带的工具集，仅本次生效
        request = request.override(tools=tools)
        return handler(request)

class ToolGuard(AgentMiddleware):
    def wrap_tool_call(
        self,
        request: ToolCallRequest,
        handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]],
    ) -> ToolMessage | Command[Any]:
        store = request.runtime.store
        context = request.runtime.context
        username = context.username
        value = store.get(namespace, username).value
        tool_name = request.tool.name

        # 本次调用花费
        credits_cost = CREDIT_MAP.get(tool_name)
        # 剩余额度
        credits_left = value["tokens_left"]
        if credits_left < credits_cost:
            logger.warning(f"{username} 额度不足，{tool_name} 调用失败！")
            return Command(
                update={
                    "messages": [
                        ToolMessage(
                            content="额度不足，无法调用工具，请充值~",
                            tool_call_id=request.runtime.tool_call_id
                        )
                    ]
                }
            )
        # 更新剩余额度
        credits_left -= credits_cost
        value["tokens_left"] = credits_left
        store.put(namespace, username, value)

        logger.info(f"{username} 调用 {tool_name} 花费 {credits_cost}，剩余额度 {credits_left}")
        logger.info(f"{username} 状态已更新，当前状态 {store.get(namespace,username).value}")
        return handler(request)

agent = create_agent(
    model = model,
    middleware=[CheckCredit(), ToolGuard()],
    store=store,
    tools=[get_weather, get_news],
    context_schema=UserContext
)

response1 = agent.invoke(
    {"messages": ["你好啊，今天北京天气如何，今日有哪些新闻？"]},
    context=UserContext(username="Ada Lovelace")
)
print("=" * 30, '-> Ada Lovelace messages <-', '=' * 30)
for msg in response1["messages"]:
    msg.pretty_print()

2026-11-23 22:53:30.321 +08:00 | WARNING  | __main__:wrap_model_call:86 - Ada Lovelace 无权调用 get_news
2026-11-23 22:53:31.691 +08:00 | INFO     | __main__:wrap_tool_call:125 - Ada Lovelace 调用 get_weather 花费 10，剩余额度 2990
2026-11-23 22:53:31.693 +08:00 | INFO     | __main__:wrap_tool_call:126 - Ada Lovelace 状态已更新，当前状态 {'tokens_left': 2990, 'get_weather': 'yes', 'get_news': 'no'}
2026-11-23 22:53:31.696 +08:00 | WARNING  | __main__:wrap_model_call:86 - Ada Lovelace 无权调用 get_news


============================== -> Ada Lovelace messages <- ==============================
================================ Human Message =================================

你好啊，今天北京天气如何，今日有哪些新闻？
================================== Ai Message ==================================

你好！我来帮你查一下北京的天气。
Tool Calls:
  get_weather (call_00_qdl0yahmmzqepxsBqF5b3407)
 Call ID: call_00_qdl0yahmmzqepxsBqF5b3407
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

北京 今天天气不错
================================== Ai Message ==================================

你好！关于你问的两件事：

**1. 北京天气**

我调用天气查询工具后，只得到了一句很简略的结果：**“北京今天天气不错”**。这个接口没有返回具体温度、湿度或风力等细节，所以只能参考大致判断：今天北京整体天气条件应该是比较好的。出门前建议你再看一眼手机上的天气 App，确认实时温度、空气质量和是否需要防晒/带伞。

**2. 今日新闻**

这部分我目前帮不上：我手上只有一个天气查询工具，**没有联网获取新闻的能力**，也无法确认“今天”具体发生了哪些事，所以我不愿意凭记忆给你讲可能已经过时或错误的“新闻”。

建议你通过以下渠道查看今日新闻：
- 主流新闻客户端（新华社、人民日报、央视新闻等官方账号）
- 门户网站首页的实时资讯流
- 财经类：财新、第一财经、彭博/路透中文

如果你愿意告诉我你关注的方向（比如科技、财经、体育、本地民生），我可以帮你梳理需要留意